# Weekly demand remains heterogeneous

Weekly aggregation reduces the sparsity problem seen at the daily product-store grain, but it does **not** turn the data into one homogeneous forecasting problem. Product-store series still split into smooth, erratic, intermittent, and lumpy demand patterns.

This matters for baseline and model design: a single forecasting method is unlikely to be equally appropriate for all product-store series. Smooth and erratic series can be handled with regular time-series baselines, while intermittent and lumpy series need sparse-demand baselines or two-stage approaches.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import duckdb
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 7)

DATA_DIR_CANDIDATES = [
    Path("../../data/processed/transactions_dst_over_weeks"),
    Path("../data/processed/transactions_dst_over_weeks"),
    Path("data/processed/transactions_dst_over_weeks"),
]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find data/processed/transactions_dst_over_weeks. "
        "Run src/data/preparation/distribute_sales_over_active_weeks.py first."
    )

parquet_glob = str(DATA_DIR / "*.parquet")
print(f"Using weekly data from {DATA_DIR}")

## Example selection

The examples below are selected to illustrate the four weekly demand clusters across multiple products:

- `PRODUCT_EXAMPLE_COUNT` products, prioritizing products that appear in all four demand classes across different stores;
- one product-store series per available product and demand class, shown in a 4 x 4 product-by-class grid;
- at least `MIN_DEMAND_WEEKS` non-zero demand weeks, so the pattern is not driven by one-off observations;
- ADI and CV² close to the class median;
- stable article metadata within the series (`ARTIKEL_INHALT`, `VERKAUFSEINHEIT`, `GEWICHTSARTIKEL` each have one value);
- longer active histories are preferred, but not required, because the available date range can differ by data extract.

The selection ranks stable-metadata articles by demand-class coverage first and class-median distance second. This keeps the focus on how the same product can produce different store-level demand patterns, while making any unavailable product/class combinations explicit in the plot.

In [ ]:
GROUP_COLS = ["ARTIKEL_ID", "MARKT_ID"]
DEMAND_COL = "ABVERKAUFTE_MENGE_KG"
MIN_DEMAND_WEEKS = 10
PRODUCT_EXAMPLE_COUNT = 4
PREFERRED_MIN_ACTIVE_WEEKS = 100
ADI_CUTOFF = 1.32
CV2_CUTOFF = 0.49
CLASS_ORDER = ["smooth", "erratic", "intermittent", "lumpy"]
CLASS_PALETTE = {
    "smooth": "#2a9d8f",
    "erratic": "#e9c46a",
    "intermittent": "#457b9d",
    "lumpy": "#d62828",
}
CLASS_LABELS = {
    "smooth": "Smooth",
    "erratic": "Erratic",
    "intermittent": "Intermittent",
    "lumpy": "Lumpy",
}

metrics_query = f"""
WITH period_data AS (
    SELECT
        ARTIKEL_ID,
        MARKT_ID,
        CAST(DATE AS DATE) AS week_start,
        SUM(CAST(COALESCE({DEMAND_COL}, 0) AS DOUBLE)) AS demand,
        MIN(ARTIKEL_BEZ) AS ARTIKEL_BEZ,
        MIN(ARTIKEL_INHALT) AS ARTIKEL_INHALT,
        MIN(VERKAUFSEINHEIT) AS VERKAUFSEINHEIT,
        MIN(GEWICHTSARTIKEL) AS GEWICHTSARTIKEL
    FROM read_parquet(?)
    GROUP BY ARTIKEL_ID, MARKT_ID, week_start
), series AS (
    SELECT
        ARTIKEL_ID,
        MARKT_ID,
        COUNT(*) AS active_weeks,
        SUM(CASE WHEN demand > 0 THEN 1 ELSE 0 END) AS demand_weeks,
        SUM(demand) AS total_demand,
        AVG(CASE WHEN demand > 0 THEN demand END) AS mean_nonzero_demand,
        VAR_SAMP(CASE WHEN demand > 0 THEN demand END) AS var_nonzero_demand,
        MIN(week_start) AS first_active_week,
        MAX(week_start) AS last_active_week,
        MIN(ARTIKEL_BEZ) AS ARTIKEL_BEZ,
        MIN(ARTIKEL_INHALT) AS ARTIKEL_INHALT,
        MIN(VERKAUFSEINHEIT) AS VERKAUFSEINHEIT,
        MIN(GEWICHTSARTIKEL) AS GEWICHTSARTIKEL,
        COUNT(DISTINCT ARTIKEL_INHALT) AS n_artikel_inhalt,
        COUNT(DISTINCT VERKAUFSEINHEIT) AS n_verkaufseinheit,
        COUNT(DISTINCT GEWICHTSARTIKEL) AS n_gewichtsartikel
    FROM period_data
    GROUP BY ARTIKEL_ID, MARKT_ID
)
SELECT
    *,
    active_weeks / NULLIF(demand_weeks, 0) AS ADI,
    CASE
        WHEN demand_weeks > 1 AND mean_nonzero_demand > 0
            THEN var_nonzero_demand / (mean_nonzero_demand * mean_nonzero_demand)
        WHEN demand_weeks = 1 THEN 0.0
        ELSE NULL
    END AS CV2
FROM series
WHERE demand_weeks >= {MIN_DEMAND_WEEKS}
"""

con = duckdb.connect()
con.execute("PRAGMA threads=4")
series_metrics = con.execute(metrics_query, [parquet_glob]).fetchdf()

conditions = [
    (series_metrics["ADI"] < ADI_CUTOFF) & (series_metrics["CV2"] < CV2_CUTOFF),
    (series_metrics["ADI"] < ADI_CUTOFF) & (series_metrics["CV2"] >= CV2_CUTOFF),
    (series_metrics["ADI"] >= ADI_CUTOFF) & (series_metrics["CV2"] < CV2_CUTOFF),
    (series_metrics["ADI"] >= ADI_CUTOFF) & (series_metrics["CV2"] >= CV2_CUTOFF),
]
choices = ["smooth", "erratic", "intermittent", "lumpy"]

series_classification = series_metrics.assign(
    demand_class=np.select(conditions, choices, default="unclassified")
)
series_classification = series_classification[
    series_classification["demand_class"].isin(CLASS_ORDER)
].copy()

class_medians = (
    series_classification
    .groupby("demand_class", observed=True)[["ADI", "CV2"]]
    .median()
    .rename(columns={"ADI": "class_median_ADI", "CV2": "class_median_CV2"})
)

class_summary = (
    series_classification
    .groupby("demand_class", observed=True)
    .agg(
        series=("ARTIKEL_ID", "size"),
        median_ADI=("ADI", "median"),
        median_CV2=("CV2", "median"),
        median_active_weeks=("active_weeks", "median"),
        median_demand_weeks=("demand_weeks", "median"),
    )
    .reindex(CLASS_ORDER)
)
class_summary["series_share_%"] = 100 * class_summary["series"] / class_summary["series"].sum()
class_summary.round(2)

In [ ]:
stable_metadata = (
    (series_classification["n_artikel_inhalt"] == 1)
    & (series_classification["n_verkaufseinheit"] == 1)
    & (series_classification["n_gewichtsartikel"] == 1)
)
metadata_candidates = series_classification[stable_metadata].copy()


def score_candidates(candidates):
    scored = candidates.merge(class_medians, on="demand_class", how="left").copy()
    scored["median_distance"] = (
        (scored["ADI"] - scored["class_median_ADI"]).abs() / scored["class_median_ADI"].replace(0, np.nan)
        + (scored["CV2"] - scored["class_median_CV2"]).abs() / scored["class_median_CV2"].replace(0, np.nan)
    )
    return scored


def has_all_classes(candidates):
    return set(candidates["demand_class"].dropna()).issuperset(CLASS_ORDER)


def best_series_per_article_class(candidates):
    return (
        score_candidates(candidates)
        .sort_values(
            ["ARTIKEL_ID", "demand_class", "median_distance", "active_weeks", "demand_weeks"],
            ascending=[True, True, True, False, False],
        )
        .groupby(["ARTIKEL_ID", "demand_class"], observed=True)
        .head(1)
    )


def select_article_grid(candidates, n_products=PRODUCT_EXAMPLE_COUNT):
    if candidates.empty or not has_all_classes(candidates):
        return pd.DataFrame()

    best_per_article_class = best_series_per_article_class(candidates)

    article_scores = (
        best_per_article_class
        .groupby("ARTIKEL_ID", observed=True)
        .agg(
            n_classes=("demand_class", "nunique"),
            total_distance=("median_distance", "sum"),
            mean_distance=("median_distance", "mean"),
            median_active_weeks=("active_weeks", "median"),
            median_demand_weeks=("demand_weeks", "median"),
            ARTIKEL_BEZ=("ARTIKEL_BEZ", "first"),
            ARTIKEL_INHALT=("ARTIKEL_INHALT", "first"),
        )
        .sort_values(
            ["n_classes", "mean_distance", "median_active_weeks", "median_demand_weeks"],
            ascending=[False, True, False, False],
        )
    )

    if len(article_scores) < n_products:
        return pd.DataFrame()

    selected_article_ids = article_scores.head(n_products).index
    return (
        best_per_article_class[best_per_article_class["ARTIKEL_ID"].isin(selected_article_ids)]
        .assign(
            _article_order=lambda df: pd.Categorical(df["ARTIKEL_ID"], categories=selected_article_ids, ordered=True),
            _class_order=lambda df: pd.Categorical(df["demand_class"], categories=CLASS_ORDER, ordered=True),
        )
        .sort_values(["_article_order", "_class_order"])
        .drop(columns=["_article_order", "_class_order"])
        .reset_index(drop=True)
    )


candidate_pools = [
    (
        f"{PRODUCT_EXAMPLE_COUNT} stable-metadata products ranked by class coverage, active_weeks >= "
        f"{PREFERRED_MIN_ACTIVE_WEEKS}",
        metadata_candidates[metadata_candidates["active_weeks"] >= PREFERRED_MIN_ACTIVE_WEEKS],
        select_article_grid,
    ),
    (
        f"{PRODUCT_EXAMPLE_COUNT} stable-metadata products ranked by class coverage",
        metadata_candidates,
        select_article_grid,
    ),
    (f"{PRODUCT_EXAMPLE_COUNT} products ranked by class coverage", series_classification, select_article_grid),
]

for selection_note, candidates, selector in candidate_pools:
    example_series = selector(candidates)
    if not example_series.empty:
        break
else:
    available = series_classification.groupby("demand_class", observed=True).size().reindex(CLASS_ORDER).fillna(0).astype(int)
    available_articles = series_classification["ARTIKEL_ID"].nunique()
    raise ValueError(
        f"Could not select {PRODUCT_EXAMPLE_COUNT} articles for the weekly demand grid. "
        f"Available articles: {available_articles}. "
        f"Available series by class: {available.to_dict()}"
    )

example_series = example_series.copy()
product_lookup = example_series[["ARTIKEL_ID", "ARTIKEL_BEZ"]].drop_duplicates("ARTIKEL_ID").reset_index(drop=True)
product_lookup["product_label"] = [f"Produkt {chr(65 + i)}" for i in range(len(product_lookup))]
example_series = example_series.merge(product_lookup[["ARTIKEL_ID", "product_label"]], on="ARTIKEL_ID", how="left")
example_series["store_label"] = (
    example_series.groupby("product_label", sort=False).cumcount().map(lambda i: f"Filiale {chr(65 + i)}")
)
coverage_by_product = example_series.groupby("product_label", sort=False)["demand_class"].nunique()

example_display_cols = [
    "product_label", "demand_class", "store_label", "ARTIKEL_BEZ", "ARTIKEL_INHALT",
    "VERKAUFSEINHEIT", "GEWICHTSARTIKEL", "active_weeks", "demand_weeks",
    "ADI", "class_median_ADI", "CV2", "class_median_CV2", "median_distance",
]

print(f"Selection: {selection_note}")
print(
    f"Complete product rows: {coverage_by_product.eq(len(CLASS_ORDER)).sum()} / {PRODUCT_EXAMPLE_COUNT}; "
    f"available plotted series: {len(example_series)} / {PRODUCT_EXAMPLE_COUNT * len(CLASS_ORDER)}"
)
example_series[example_display_cols].round({
    "ADI": 2,
    "class_median_ADI": 2,
    "CV2": 2,
    "class_median_CV2": 2,
    "median_distance": 3,
})

## Vier produktbezogene Plots der wöchentlichen Nachfrage

Die ausgewählten Beispiele priorisieren vier Artikel mit möglichst breiter Abdeckung der Nachfrageklassen. Jedes Produkt wird als eigener Plot mit vier Teilplots dargestellt; nicht verfügbare Produkt-Klassen-Kombinationen werden direkt markiert.

In [ ]:
example_keys = example_series[["product_label", "demand_class", "store_label", "ARTIKEL_ID", "MARKT_ID"]].copy()
con.register("example_keys", example_keys)

examples_query = f"""
SELECT
    k.product_label,
    k.demand_class,
    k.store_label,
    w.ARTIKEL_ID,
    w.MARKT_ID,
    CAST(w.DATE AS DATE) AS week_start,
    CAST(COALESCE(w.{DEMAND_COL}, 0) AS DOUBLE) AS demand
FROM read_parquet(?) AS w
JOIN example_keys AS k
    ON w.ARTIKEL_ID = k.ARTIKEL_ID
    AND w.MARKT_ID = k.MARKT_ID
ORDER BY k.product_label, k.demand_class, k.store_label, week_start
"""
example_ts = con.execute(examples_query, [parquet_glob]).fetchdf()

plot_data = example_ts.merge(
    example_series[[
        "product_label", "demand_class", "store_label", "ARTIKEL_ID", "MARKT_ID", "ARTIKEL_BEZ",
        "ARTIKEL_INHALT", "VERKAUFSEINHEIT", "ADI", "CV2", "active_weeks", "demand_weeks"
    ]],
    on=["product_label", "demand_class", "store_label", "ARTIKEL_ID", "MARKT_ID"],
    how="left",
)

product_order = example_series[["product_label", "ARTIKEL_ID"]].drop_duplicates()["product_label"].tolist()
product_names = example_series.groupby("product_label", sort=False)["ARTIKEL_BEZ"].first().to_dict()


def shorten_label(value, max_chars=36):
    value = str(value)
    return value if len(value) <= max_chars else value[: max_chars - 3] + "..."


for product_label in product_order:
    fig, axes = plt.subplots(2, 2, figsize=(14, 7.5), sharex=False, sharey=False)
    axes = axes.ravel()

    for panel_idx, (ax, demand_class) in enumerate(zip(axes, CLASS_ORDER)):
        panel_row, panel_col = divmod(panel_idx, 2)
        class_label = CLASS_LABELS[demand_class]
        subset = plot_data[
            (plot_data["product_label"] == product_label)
            & (plot_data["demand_class"] == demand_class)
        ].sort_values("week_start")
        if subset.empty:
            ax.set_title(
                f"{product_label} · {class_label}\n"
                f"{shorten_label(product_names[product_label])}\n"
                "Keine passende Zeitreihe",
                fontsize=9,
            )
            ax.text(0.5, 0.5, "Keine Zeitreihe\nin dieser Klasse", ha="center", va="center", transform=ax.transAxes)
            ax.set_xlabel("Woche" if panel_row == 1 else "")
            ax.set_ylabel("Menge" if panel_col == 0 else "")
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        meta = subset.iloc[0]
        color = CLASS_PALETTE[demand_class]

        ax.plot(subset["week_start"], subset["demand"], color=color, linewidth=1.2)
        ax.scatter(subset["week_start"], subset["demand"], color=color, s=7, alpha=0.65)
        ax.axhline(0, color="#444444", linewidth=0.8, alpha=0.5)
        ax.set_title(
            f"{product_label} · {class_label}\n"
            f"{shorten_label(meta['ARTIKEL_BEZ'])}\n"
            f"{meta['store_label']} · ADI={meta['ADI']:.2f}, CV²={meta['CV2']:.2f}",
            fontsize=9,
        )
        ax.set_xlabel("Woche" if panel_row == 1 else "")
        ax.set_ylabel(f"Menge, kg" if panel_col == 0 else "")
        ax.tick_params(axis="x", rotation=30, labelsize=8)
        ax.tick_params(axis="y", labelsize=8)

    fig.suptitle(f"{product_label}: {shorten_label(product_names[product_label], max_chars=70)}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Interpretation

The weekly aggregation makes the data more usable for baseline modeling, but the four product-level plots show that the demand-generating process remains heterogeneous when the same product is followed across stores:

- **Smooth**: frequent demand with comparatively stable non-zero quantities.
- **Erratic**: frequent demand but volatile quantities.
- **Intermittent**: many no-sale weeks, with relatively stable non-zero quantities when sales occur.
- **Lumpy**: many no-sale weeks and volatile non-zero quantities.

This supports implementing and evaluating different model families by demand pattern instead of forcing one baseline or one forecasting model across all product-store series.